In [ ]:
#Step 12 — Feature Entanglement (visualise learned features using PCA)

# Reload the trained model before feature extraction
model = keras.models.load_model("trained_model.keras")
print("Model reloaded and ready for feature extraction.")

# Collect up to 5000 samples for speed, use a subset of data
X_all = np.vstack([X_train[:5000], X_test[:5000]])
y_all = np.concatenate([y_train[:5000], y_test[:5000]])

# --- Get the 64-unit Dense layer (the second last layer) ---
hidden_layer = model.layers[-2]

# --- Compute activations manually ---
# Pass the data through all layers up to the hidden layer
get_hidden_output = keras.Sequential(model.layers[:-1])

# Get hidden representations
H = get_hidden_output.predict(X_all, batch_size=256)

# Reduce 64D vectors to 2D using PCA
pca = PCA(n_components=2, random_state=SEED)
H2 = pca.fit_transform(H)

# Plot the 2D projection
scatter = plt.scatter(H2[:,0], H2[:,1], c=y_all, cmap="coolwarm", alpha=0.6)
plt.title("Feature Entanglement (PCA of hidden activations)")
plt.xlabel("PC1")
plt.ylabel("PC2")
cbar = plt.colorbar(scatter, ticks=[0,1])
cbar.ax.set_yticklabels(["Medical (0)", "Non-medical (1)"])
plt.tight_layout()
plt.show()

In [ ]:
# Step 12 — Feature Entanglement (UMAP + Confidence Visualization)

# --- Load trained model (if Colab restarted) ---
model = keras.models.load_model("trained_model.keras")

# --- Use all layers except final output to get hidden features ---
get_hidden_output = keras.Sequential(model.layers[:-1])

# --- Collect data subset (adjust for speed/memory) ---
X_all = np.vstack([X_train[:5000], X_test[:5000]])
y_all = np.concatenate([y_train[:5000], y_test[:5000]])

# --- Extract hidden activations from trained network ---
H = get_hidden_output.predict(X_all, batch_size=256)

# Apply UMAP for better separation
# Scale features first
H_std = StandardScaler().fit_transform(H)

# Run UMAP
um = umap.UMAP(
    n_neighbors=30,   # 15–50; higher = smoother global structure
    min_dist=0.05,    # smaller = tighter clusters
    metric='cosine',
    #random_state=SEED
)
H2 = um.fit_transform(H_std)


# Optional: color by model confidence instead of class
# Predict probabilities (confidence of class 1)
probs = model.predict(X_all, batch_size=256).ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# (A) Plot by true labels
sc1 = axes[0].scatter(H2[:,0], H2[:,1], c=y_all, cmap="coolwarm", alpha=0.65)
axes[0].set_title("UMAP of Hidden Activations (colored by True Label)")
axes[0].set_xlabel("UMAP Dim 1")
axes[0].set_ylabel("UMAP Dim 2")
cbar1 = plt.colorbar(sc1, ax=axes[0], ticks=[0,1])
cbar1.ax.set_yticklabels(["Medical (0)", "Non-medical (1)"])

# (B) Plot by model confidence
sc2 = axes[1].scatter(H2[:,0], H2[:,1], c=probs, cmap="coolwarm", alpha=0.7)
axes[1].set_title("UMAP of Hidden Activations (colored by Confidence)")
axes[1].set_xlabel("UMAP Dim 1")
axes[1].set_ylabel("UMAP Dim 2")
cbar2 = plt.colorbar(sc2, ax=axes[1])
cbar2.set_label("Confidence  (Non-medical → Medical)")

plt.tight_layout()
plt.show()